# The matched sKF with $\chi_t'$ clipped at zero

Notebook 09 (Ignacio, `ignacio/joint-vs-marginal-vs-minorized`) found that the sKF matched to the true
noise (generalized Gaussian, $\beta^* = 0.2$) recovers from a sign flip of the response **faster** than the
two Laplacian sKFs: 5979 steps against 7584 (joint) and 7340 (minorized).

The matched density is not log-concave, so $\chi_t'$ can be negative. Then eq. (36),
$v_t = \tilde v_t(1 - \chi_t'/M)$, makes $v_t > \tilde v_t$: the filter **inflates** its variance. After a
change the errors are large, and large errors are where $\chi_t' < 0$. So one possible reason for the
faster recovery is that inflation: a larger $v_t$, a larger gain, a faster recovery.

**The test.** The same filter with $\chi_t' \to \max(\chi_t', 0)$ in the variance update only. The mean
update is untouched. If the clipped filter recovers as slowly as the Laplacian ones, the inflation was the
reason; if it recovers about as fast as the unclipped one, it was not.

Issue #8 already notes that the fKF never uses $\chi_t'$ and the matched fKF still recovers faster at the
3 dB level (`fkf.ipynb`), so for the fKF clipping cannot be the reason. This notebook is about the sKF.

**Setup:** notebook 09's, unchanged. AR($-0.9$) input, room response of section 7.1, $M = 128$, 5 dB,
$b_\eta = \mathrm{E}|\eta_t|$, $R = 20$, $N = 96000$, target $-20$ dB, $\varepsilon$ by notebook 09's grid
protocol, then $2N$ steps with $\boldsymbol{h}_o \to -\boldsymbol{h}_o$ at $t = N$; recovered = first step
after the change within 3 dB of the floor at rest.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Notebook 09's, value for value: same constants, same seeds, so the same signals.

In [ ]:
# Values of notebook 09 (branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb,
# commit 6e6dabc), which copies them from notebook 07 (commit f26ed36).
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
AR_A = -0.9                 # AR(1) coefficient of the input
SNR_DB = 5.0                # nominal signal-to-noise ratio
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
VAR_THETA_0 = 2.0           # initial prior variance on each weight
FS = 8000                   # sampling rate [Hz]
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]

ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM,
                  reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)

lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))
P_signal = ho @ AR_A**lags @ ho                    # signal power, unit-variance AR(1) input
var_eta = P_signal/10**(SNR_DB/10)                 # noise variance that gives SNR_DB
scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))   # generalized Gaussian scale
b_eta = scale_gg*np.exp(gammaln(2/BETA) - gammaln(1/BETA))               # E|eta|, the Laplacian scale
w0 = np.zeros(M)


# === IGNACIO: generate_signals - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
def generate_signals(seed):
    """One realisation: AR(1) input, its clean output through ho, and heavy-tailed noise."""
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - AR_A**2)*rng.standard_normal(N + WARMUP)   # driving noise, unit-variance x
    x = lfilter([1.0], [1.0, -AR_A], u)[WARMUP:]               # x_t = AR_A x_{t-1} + u_t
    y = np.convolve(ho, x)[:N]                                 # clean output, y_t = x_t' ho
    eta = gennorm.rvs(BETA, scale=scale_gg, size=N, random_state=rng)
    return x, y + eta, eta
# === end of the copied block ===


print(f"var_eta = {var_eta:.4e}, noise scale = {scale_gg:.4e}, b_eta = E|eta| = {b_eta:.4e}")

## 3. The filters

Notebook 09's three robust sKFs, copied verbatim, used only for the checks of section 4. The runs use a
batched form (one filter per row), from `fkf.ipynb`, with a `clip` option added. The quadrature of
notebook 08 takes one error at a time; here it is written over arrays, with the same pieces, nodes and
weights, and checked against the original.

| filter | $\chi_t$ | in $v_t$ |
|---|---|---|
| minorized | $\tau_t e_t/(\tau_t + \lvert e_t\rvert)$, eq. (45) | $\chi_t/e_t$ |
| joint (exact Laplacian) | $\tau_t\Lambda_t$, eq. (43) | $\chi_t'$ |
| matched | mean of the scalar posterior (21), quadrature | $\chi_t' = 1 - \mathrm{Var}[s \mid y_{1:t}]/\sigma_t^2$ |
| matched, clipped | the same | $\max(\chi_t', 0)$ |

In [ ]:
# === IGNACIO: notebook 09's filters and quadrature - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
# (sKF-L filters and chi_laplacian from notebook 07, f26ed36; quadrature from notebook 08, 0ed92a4)
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def sKF_L_minorized(n, x, d, w0, parameters):
    """Eqs. (robust.sKF.mean) and (robust.sKF.variance): the sKF above with v_eta -> b_eta |e_t|."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, only for k_t

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            power = x_t @ x_t                          # ||x_t||^2
            denominator = b_eta*abs(e[k]) + v_tilde*power     # b_eta |e_t| + v_tilde ||x_t||^2
            w = w + x_t*(v_tilde*e[k]/denominator)     # w = w + v_tilde x_t e_t / denominator
            v = v_tilde*(1 - v_tilde*power/(M*denominator))   # v = v_tilde (1 - v_tilde ||x||^2/(M den))
            sigma_hist[k] = np.sqrt(v_tilde*power)     # sigma_t, eq. (16), recorded for k_t only

    return {"w_hist": w_hist, "e": e, "sigma": sigma_hist}


def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope


def sKF_L_joint(n, x, d, w0, parameters):
    """Eqs. (35) and (36) of the new draft with the Laplacian chi of eq. (43): joint posterior mean."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, only for k_t

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            power = x_t @ x_t                          # ||x_t||^2
            sigma = np.sqrt(v_tilde*power)             # sigma_t = sqrt(v_tilde ||x_t||^2), eq. (16)
            chi, chi_slope = chi_laplacian(e[k], sigma, b_eta)
            w = w + x_t*chi/power                      # w = w + x_t chi / ||x_t||^2, eq. (35)
            v = v_tilde*(1 - chi_slope/M)              # v = v_tilde (1 - chi'/M), eq. (36)
            sigma_hist[k] = sigma                      # recorded for k_t only

    return {"w_hist": w_hist, "e": e, "sigma": sigma_hist}


def gg_scale(beta, var_eta):
    """alpha(beta): the generalized Gaussian of shape beta closest in KL to the simulated noise."""
    alpha_true = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))                 # alpha_*
    mean_abs_pow = alpha_true**beta*np.exp(gammaln((beta + 1)/BETA) - gammaln(1/BETA))    # E|eta|^beta
    return (beta*mean_abs_pow)**(1/beta)                                                   # alpha(beta)


L_WINDOW = 10               # the prior N(s; 0, sigma^2) is below e^{-50} beyond L_WINDOW sigma


def log_likelihood(u, density, params):
    """log p_eta(u) up to a constant: the noise density the filter assumes, in eqs. (21) and (24)."""
    if density == "gg":                                # generalized Gaussian, shape beta, scale alpha
        return -(np.abs(u)/params["alpha"])**params["beta"]
    # Student-t, nu degrees of freedom and scale c: -(nu + 1)/2 log(1 + (u/c)^2/nu)
    return -0.5*(params["nu"] + 1)*np.log1p((u/params["scale"])**2/params["nu"])


def piece_in_s(e, s_a, s_b, density, params):
    """Gauss-Legendre nodes on [s_a, s_b], directly in s. Returns the nodes and the log of
    (weight x length x likelihood p_eta(e_t - s)), the prior left out."""
    length = s_b - s_a                                 # length of the piece
    s = s_a + length*params["nodes"]                   # nodes mapped from [0, 1] to [s_a, s_b]
    log_q = params["log_weights"] + np.log(length) + log_likelihood(e - s, density, params)
    return s, log_q


def piece_in_z(e, s_a, s_b, params):
    """Gauss-Legendre nodes on [s_a, s_b] in the variable z = (|e_t - s|/alpha)^beta, where the
    cusp of the likelihood at s = e_t becomes the smooth e^{-z}. Same output as piece_in_s."""
    alpha = params["alpha"]
    beta = params["beta"]
    side = 1.0 if s_a >= e else -1.0                   # the piece lies above or below e_t
    z_a = (abs(e - s_a)/alpha)**beta                   # z at the two ends of the piece
    z_b = (abs(e - s_b)/alpha)**beta
    z_low = min(z_a, z_b)
    length = max(z_a, z_b) - z_low                     # length of the piece in z
    z = z_low + length*params["nodes"]                 # nodes mapped from [0, 1] to the piece
    log_z = np.log(z)
    s = e + side*alpha*np.exp(log_z/beta)              # s = e_t + side alpha z^(1/beta)
    # log of weight x length x ds/dz x likelihood, with ds/dz = (alpha/beta) z^(1/beta - 1)
    # and the likelihood e^{-z}
    log_q = params["log_weights"] + np.log(length*alpha/beta) + (1/beta - 1)*log_z - z
    return s, log_q


def chi_quadrature(e, sigma, density, params):
    """chi_t(e_t) and chi'_t(e_t) of eq. (30), from the mean and variance of the scalar
    posterior (21), by Gauss-Legendre quadrature. Same output as chi_laplacian."""
    # Cuts at the peak of the prior (0), the peak of the likelihood (e_t) and the ends of the
    # prior (+-L sigma), so that both peaks sit at the ends of pieces, where the nodes crowd.
    cuts = sorted([-L_WINDOW*sigma, 0.0, e, L_WINDOW*sigma])
    s_all = []                                         # nodes in s of every piece
    log_q_all = []                                     # log of their weights, eq. (21) times ds
    for piece in range(3):
        s_a = cuts[piece]
        s_b = cuts[piece + 1]
        if s_b <= s_a:                                 # e_t on a cut: this piece is empty
            continue
        if density == "gg" and params["beta"] < 1:
            s, log_q = piece_in_z(e, s_a, s_b, params)
        else:
            s, log_q = piece_in_s(e, s_a, s_b, density, params)
        s_all.append(s)
        log_q_all.append(log_q - s*s/(2*sigma**2))     # times the prior N(s; 0, sigma^2), eq. (21)
    s = np.concatenate(s_all)
    log_q = np.concatenate(log_q_all)
    q = np.exp(log_q - log_q.max())                    # largest weight becomes 1, nothing underflows
    total = q.sum()
    mean = (q @ s)/total                               # E[s_t | y_1:t]
    var = (q @ (s - mean)**2)/total                    # Var[s_t | y_1:t]
    return mean, 1 - var/sigma**2                      # chi and chi', eq. (30)


def density_params(density, shape, scale, n_nodes):
    """Everything chi_quadrature needs about one density: shape, scale and the nodes."""
    nodes, weights = np.polynomial.legendre.leggauss(n_nodes)   # on [-1, 1], computed once
    nodes = (nodes + 1)/2                              # moved to [0, 1]: a piece [a, b] takes
    weights = weights/2                                # a + (b - a) node, weight (b - a) weight
    if density == "gg":
        return {"beta": shape, "alpha": scale, "nodes": nodes, "log_weights": np.log(weights)}
    return {"nu": shape, "scale": scale, "nodes": nodes, "log_weights": np.log(weights)}


def sKF_quadrature(n, x, d, w0, parameters):
    """Eqs. (35) and (36) of the new draft with chi and chi' by quadrature, eqs. (21) and (30)."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    density = parameters["density"]                    # "gg" or "t", the noise the filter assumes
    params = parameters["density_params"]              # its shape, its scale and the nodes
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, for k_t
    chi_slope_hist = np.zeros((n,))                    # chi'_t at every step, for the guard

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            power = x_t @ x_t                          # ||x_t||^2
            sigma = np.sqrt(v_tilde*power)             # sigma_t = sqrt(v_tilde ||x_t||^2), eq. (16)
            chi, chi_slope = chi_quadrature(e[k], sigma, density, params)   # eqs. (21) and (30)
            assert chi_slope <= 1 + 1e-9, f"chi' = {chi_slope} > 1 at step {k}"   # Var[s|y] >= 0
            w = w + x_t*chi/power                      # w = w + x_t chi / ||x_t||^2, eq. (35)
            v = v_tilde*(1 - chi_slope/M)              # v = v_tilde (1 - chi'/M), eq. (36)
            assert np.isfinite(v) and np.isfinite(w).all(), f"not finite at step {k}"
            sigma_hist[k] = sigma
            chi_slope_hist[k] = chi_slope

    return {"w_hist": w_hist, "e": e, "sigma": sigma_hist, "chi_slope": chi_slope_hist}
# === end of the copied block ===

In [ ]:
# Copied from fkf.ipynb, commit 058a9f9.
def chi_minorized(e, sigma, b_eta):
    """Eq. (45) of the draft (chi.minorized): the minorized correction tau e/(tau + |e|), and the
    ratio chi/e that replaces chi' in the variance update (Table 2). The fKF uses the first only."""
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta
    ratio = tau/(tau + np.abs(e))                      # chi_min / e, in (0, 1]
    return ratio*e, ratio


def vectorised_gg(params):
    """chi_quadrature of notebook 08 for the generalized Gaussian with beta < 1, over arrays of
    errors at once: the same three pieces, the same change of variable (piece_in_z), the same nodes
    and weights. New here; checked against chi_quadrature below. b_eta is not used."""
    alpha, beta = params["alpha"], params["beta"]
    nodes, log_weights = params["nodes"], params["log_weights"]

    def chi_fn(e, sigma, b_eta=None):
        e = np.asarray(e, dtype=float)
        sigma = np.broadcast_to(np.asarray(sigma, dtype=float), e.shape)
        # cuts at -L sigma, 0, e_t and +L sigma, sorted: three pieces per error
        cuts = np.sort(np.stack([-L_WINDOW*sigma, np.zeros_like(e), e, L_WINDOW*sigma], axis=-1), axis=-1)
        s_a, s_b = cuts[..., :3], cuts[..., 1:]
        e_col = e[..., None]
        side = np.where(s_a >= e_col, 1.0, -1.0)      # the piece lies above or below e_t
        z_a = (np.abs(e_col - s_a)/alpha)**beta        # z = (|e_t - s|/alpha)^beta at both ends
        z_b = (np.abs(e_col - s_b)/alpha)**beta
        z_low = np.minimum(z_a, z_b)
        length = np.maximum(z_a, z_b) - z_low
        with np.errstate(divide="ignore", invalid="ignore"):   # empty pieces, masked just below
            z = z_low[..., None] + length[..., None]*nodes
            log_z = np.log(z)
            s = e_col[..., None] + side[..., None]*alpha*np.exp(log_z/beta)
            log_q = (log_weights + np.log(length*alpha/beta)[..., None] + (1/beta - 1)*log_z - z
                     - s*s/(2*sigma[..., None, None]**2))     # times the prior N(s; 0, sigma^2)
        full = (s_b > s_a)[..., None]                  # e_t on a cut: that piece is empty
        log_q = np.where(full, log_q, -np.inf).reshape(e.shape + (-1,))
        s = np.where(full, s, 0.0).reshape(e.shape + (-1,))
        q = np.exp(log_q - log_q.max(axis=-1, keepdims=True))
        total = q.sum(axis=-1)
        mean = (q*s).sum(axis=-1)/total                # E[s_t | y_1:t]
        var = (q*(s - mean[..., None])**2).sum(axis=-1)/total   # Var[s_t | y_1:t]
        return mean, 1 - var/sigma**2                  # chi and chi', eq. (30)
    return chi_fn


# sKF_batch and fKF_batch: copied from fkf.ipynb, commit 058a9f9 (sKF_batch from vkf-kf.ipynb,
# commit 2724d95), with options added and nothing else changed. Defaults reproduce the originals.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    return mis


N_NODES = 100                                      # nodes per piece, as notebook 09
alpha_star = gg_scale(BETA, var_eta)               # the true scale: the matched filter knows the noise
PARAMS_GG = density_params("gg", BETA, alpha_star, N_NODES)
chi_matched = vectorised_gg(PARAMS_GG)

FILTERS = {                                        # name -> (correction, clip)
    "minorized": (chi_minorized, False),
    "joint": (chi_laplacian, False),
    "matched": (chi_matched, False),
    "matched, clipped": (chi_matched, True),
}
print("filters ready")

## 4. Checks

1. The vectorised quadrature against notebook 08's `chi_quadrature`, one error at a time.
2. The batched sKF against notebook 09's scalar filters, realisation 0, 3000 steps, for the three
   corrections of notebook 09.
3. Clipping must change nothing where $\chi_t' \ge 0$ always: the joint Laplacian filter with `clip=True`
   must equal the one without.

In [ ]:
# The vectorised quadrature against notebook 08's chi_quadrature, one error at a time: errors
# from the noise plus the prior, and far out in both tails, over the sigma the filters visit.
rng_check = np.random.default_rng(0)
n_check = 20000
sigma_check = 10**rng_check.uniform(-3, 0.5, n_check)
e_check = sigma_check*rng_check.standard_normal(n_check)*10**rng_check.uniform(-3, 2, n_check)
e_check[:n_check//2] = (gennorm.rvs(BETA, scale=scale_gg, size=n_check//2, random_state=rng_check)
                        + sigma_check[:n_check//2]*rng_check.standard_normal(n_check//2))
e_check[:3] = [0.0, 50.0, -50.0]                       # on a cut, and far past +-L sigma
start = time.time()
scalar = np.array([chi_quadrature(a, b, "gg", PARAMS_GG) for a, b in zip(e_check, sigma_check)])
us_scalar = (time.time() - start)/n_check*1e6
start = time.time()
chi_v, slope_v = chi_matched(e_check, sigma_check)
us_vector = (time.time() - start)/n_check*1e6
print(f"vectorised quadrature against chi_quadrature, {n_check} errors")
print(f"   max |chi difference|/sigma = {np.max(np.abs(chi_v - scalar[:, 0])/sigma_check):.1e}")
print(f"   max |chi' difference|      = {np.max(np.abs(slope_v - scalar[:, 1])):.1e}")
print(f"   cost per error: {us_scalar:.1f} us one at a time, {us_vector:.1f} us in one array")

N_CMP = 3000
x_0, d_0, _ = generate_signals(0)
X_0, D_0 = x_0[None, :N_CMP], d_0[None, :N_CMP]
EPS_CMP = 2e-6
print(f"\nbatched sKF against notebook 09's scalar filters, realisation 0, {N_CMP} steps, eps = {EPS_CMP:.0e}")
for name, scalar_fn, parameters in [
        ("minorized", sKF_L_minorized, {"b_eta": b_eta}),
        ("joint", sKF_L_joint, {"b_eta": b_eta}),
        ("matched", sKF_quadrature, {"density": "gg", "density_params": PARAMS_GG})]:
    parameters.update(epsilon=EPS_CMP, var_theta_0=VAR_THETA_0)
    w_hist = scalar_fn(N_CMP, x_0[:N_CMP], d_0[:N_CMP], w0, parameters)["w_hist"]
    mis_scalar = ((w_hist - ho)**2).sum(axis=1)
    mis_batch = sKF_batch(X_0, D_0, ho, b_eta, [EPS_CMP], FILTERS[name][0])[0]
    print(f"   {name:<10} max relative difference = {np.max(np.abs(mis_batch - mis_scalar)/mis_scalar):.1e}")

same = np.array_equal(sKF_batch(X_0, D_0, ho, b_eta, [EPS_CMP], chi_laplacian, clip=True),
                      sKF_batch(X_0, D_0, ho, b_eta, [EPS_CMP], chi_laplacian))
print(f"\njoint filter, clip=True against clip=False: {'identical' if same else 'DIFFERENT'}")

## 5. At rest

Notebook 09's protocol, copied: $\varepsilon$ on logspace$(-8, -3, 11)$, every grid point at $R = 20$,
$N = 96000$; floor = mean of the last quarter; a point is kept if it settled (within 3 dB of its floor by
half the run, third quarter within 0.5 dB of the floor); the $\varepsilon$ that lands on $-20$ dB is
interpolated over the kept points and run once on the same signals.

Clipping changes the floor a given $\varepsilon$ reaches, so the clipped filter gets its own search.

**Control:** the three filters of notebook 09 must land on its numbers.

In [ ]:
# === IGNACIO: steady_state and is_settled - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)
# === end of the copied block ===



TARGET_DB = -20.0
DRIFT_DB = 0.5
EPS_GRID = np.logspace(-8, -3, 11)                 # notebook 09's grid, for all four filters

sweep_signals = [generate_signals(seed)[:2] for seed in range(R)]
X_rest = np.array([s[0] for s in sweep_signals])
D_rest = np.array([s[1] for s in sweep_signals])


def sweep(chi_fn, clip):
    """sweep_filter and pick_parameter of notebook 09, with every grid point and realisation in one
    batch. Same grid, signals, settling rule and interpolation."""
    G = len(EPS_GRID)
    mis = sKF_batch(np.repeat(X_rest, G, axis=0), np.repeat(D_rest, G, axis=0), ho, b_eta,
                    np.tile(EPS_GRID, R), chi_fn, clip=clip)
    curves = mis.reshape(R, G, -1).mean(axis=0)
    del mis
    floors = np.array([steady_state(c)[0] for c in curves])
    settled = np.array([is_settled(c, DRIFT_DB) for c in curves])
    values, kept = EPS_GRID[settled], floors[settled]
    if TARGET_DB < kept.min():
        value, status = values[np.argmin(kept)], "target not reached"
    else:
        order = np.argsort(kept)
        value, status = 10**np.interp(TARGET_DB, kept[order], np.log10(values)[order]), "ok"
    curve = sKF_batch(X_rest, D_rest, ho, b_eta, np.full(R, value), chi_fn, clip=clip).mean(axis=0)
    floor, reached = steady_state(curve)
    return dict(value=value, floor=floor, reached=reached, status=status, floors=floors, settled=settled)


rest = {}
for name, (chi_fn, clip) in FILTERS.items():
    start = time.time()
    rest[name] = sweep(chi_fn, clip)
    print(f"{name:<18} done in {time.time() - start:.0f} s")

print(f"\n{'filter':<18}{'epsilon':>11}{'floor [dB]':>12}{'steps':>8}  status")
for name, r in rest.items():
    print(f"{name:<18}{r['value']:>11.4e}{r['floor']:>12.4f}{r['reached']:>8d}  {r['status']}"
          f"   ({int((~r['settled']).sum())} grid points not settled)")

# notebook 09's printed values (its section 5 and its control against notebook 07)
NB09 = {"minorized": (2.641249e-06, 4357), "joint": (1.349260e-06, 4756), "matched": (2.19e-06, 3630)}
print("\ncontrol against notebook 09:")
for name, (value, steps) in NB09.items():
    ok = np.isclose(rest[name]["value"], value, rtol=5e-3 if name == "matched" else 1e-5) \
        and rest[name]["reached"] == steps
    print(f"   {name:<10} epsilon {rest[name]['value']:.4e} / {value:.4e}, steps {rest[name]['reached']} / {steps}"
          f"   {'same' if ok else 'DIFFERENT'}")

## 6. Across the change

Notebook 09's test, copied: $2N$ steps, $\boldsymbol{h}_o \to -\boldsymbol{h}_o$ at $t = N$, each filter
at its $\varepsilon$ at rest. $\chi_t'$ and $v_t$ are recorded at every step.

**Control:** the three filters of notebook 09 must recover in its 7340, 7584 and 5979 steps.

In [ ]:
N_CHANGE = 2*N              # N steps to converge, the change, N steps to recover


# === IGNACIO: generate_signals_change and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
def generate_signals_change(seed):
    """One realisation of 2N steps: the response flips sign at step N, theta -> -theta."""
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - AR_A**2)*rng.standard_normal(N_CHANGE + WARMUP)   # driving noise, unit-variance x
    x = lfilter([1.0], [1.0, -AR_A], u)[WARMUP:]               # x_t = AR_A x_{t-1} + u_t
    y = np.convolve(ho, x)[:N_CHANGE]                          # clean output through ho
    y[N:] = -y[N:]                                             # from step N on, through -ho
    eta = gennorm.rvs(BETA, scale=scale_gg, size=N_CHANGE, random_state=rng)
    return x, y + eta, eta


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===



change_signals = [generate_signals_change(seed)[:2] for seed in range(R)]
X_change = np.array([s[0] for s in change_signals])
D_change = np.array([s[1] for s in change_signals])

change = {}
for name, (chi_fn, clip) in FILTERS.items():
    start = time.time()
    mis, slope, v = sKF_batch(X_change, D_change, ho, b_eta, np.full(R, rest[name]["value"]), chi_fn,
                              flip_at=N, clip=clip, record=True)
    change[name] = dict(curve=mis.mean(axis=0), slope=slope, v=v.mean(axis=0))
    change[name]["recovery"] = recovery(change[name]["curve"], rest[name]["floor"])
    print(f"{name:<18} done in {time.time() - start:.0f} s")

print(f"\n{'filter':<18}{'floor at rest [dB]':>19}{'before [dB]':>13}{'recovery':>10}{'time [s]':>10}")
for name in FILTERS:
    before = 10*np.log10(change[name]["curve"][3*N//4:N].mean())
    print(f"{name:<18}{rest[name]['floor']:>19.2f}{before:>13.2f}{change[name]['recovery']:>10d}"
          f"{change[name]['recovery']/FS:>10.2f}")
print("   before: mean of the last quarter before the change, same run")

print("\ncontrol against notebook 09:")
for name, steps in (("minorized", 7340), ("joint", 7584), ("matched", 5979)):
    print(f"   {name:<10} {change[name]['recovery']} / {steps}   "
          f"{'same' if change[name]['recovery'] == steps else 'DIFFERENT'}")

rec = {name: change[name]["recovery"] for name in FILTERS}
print()
for other in ("joint", "minorized"):
    print(f"matched / {other:<9}: {rec['matched']/rec[other]:.2f}x      "
          f"clipped / {other:<9}: {rec['matched, clipped']/rec[other]:.2f}x")
print(f"clipped / matched  : {rec['matched, clipped']/rec['matched']:.2f}x")

### How often $\chi_t' < 0$

Over all realisations, in windows of the change run: at rest (the last quarter before the change), and
after the change, split into its first 500 steps, the next 1500, and the rest of the way to recovery.
The clipped filter's $\chi_t'$ is the value before clipping.

In [ ]:
def windows(name):
    r = change[name]["recovery"]
    return [("at rest, last quarter before the change", 3*N//4, N),
            ("first 500 steps after the change", N, N + 500),
            ("steps 500 to 2000 after", N + 500, N + 2000),
            ("step 2000 to recovery", N + 2000, N + r),
            ("the whole recovery", N, N + r)]


for name in ("matched", "matched, clipped"):
    print(f"{name}")
    heads = ("chi' < 0", "mean chi'", "lowest chi'")
    print(f"   {'window':<42}{heads[0]:>10}{heads[1]:>11}{heads[2]:>13}")
    for label, a, b in windows(name):
        s = change[name]["slope"][:, a:b]
        print(f"   {label:<42}{np.mean(s < 0):>10.1%}{np.mean(s):>11.3f}{s.min():>13.3f}")

head = "mean chi', first 500"
print("\nv_t, mean over the realisations (for the minorized filter chi' is its ratio chi/e):")
print(f"   {'filter':<18}{'before':>11}{'largest after':>15}{'ratio':>8}{'at step':>9}{head:>22}")
for name in FILTERS:
    v = change[name]["v"]
    before = v[N - 1000:N].mean()
    print(f"   {name:<18}{before:>11.3e}{v[N:].max():>15.3e}{v[N:].max()/before:>8.2f}"
          f"{int(v[N:].argmax()):>9d}{np.mean(change[name]['slope'][:, N:N + 500]):>22.3f}")

### Recovery read at several levels

As in `fkf.ipynb` section 6: the first step after the change below each level.

In [ ]:
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0, "floor + 3"]
names = list(FILTERS)
level_steps = {}
print(f"{'level [dB]':>12}" + "".join(f"{n:>18}" for n in names)
      + f"{'matched/joint':>15}{'clipped/joint':>15}{'clipped/matched':>17}")
for level in LEVELS:
    steps = {n: (rec[n] if level == "floor + 3" else time_to(change[n]["curve"], level)) for n in names}
    level_steps[level] = steps
    text = f"{level:>12}" if isinstance(level, str) else f"{level:>12.0f}"
    print(text + "".join(f"{steps[n]:>18d}" for n in names)
          + f"{steps['matched']/steps['joint']:>15.2f}{steps['matched, clipped']/steps['joint']:>15.2f}"
          + f"{steps['matched, clipped']/steps['matched']:>17.2f}")
print("   steps after the change; b_eta = E|eta|")

**Figure 1.** Misalignment across the change, time counted from it. Dotted, in each curve's colour: the
instant it gets back within 3 dB of its floor at rest. Grey, dotted: that recovery level (the four floors
are within 0.03 dB). Black, dashed: the change.

In [ ]:
# Colours as fkf.ipynb; the clipped variant takes the violet of the same palette (checked for
# colour-blind readers against the other three: CVD Delta E >= 9.2), dashed, as a variant of the matched.
COLOUR = {"minorized": "#2a78d6", "joint": "#eb6834", "matched": "#1baf7a", "matched, clipped": "#4a3aa7"}
STYLE = {"minorized": "-", "joint": "-", "matched": "-", "matched, clipped": "--"}
t_change = (np.arange(N_CHANGE) - N)/FS
keep = (t_change >= -0.25) & (t_change <= 2.0*max(rec.values())/FS)

fig, ax = plt.subplots(figsize=(9, 4.4), constrained_layout=True)
for name in FILTERS:
    ax.plot(t_change[keep], 10*np.log10(change[name]["curve"][keep]), color=COLOUR[name], ls=STYLE[name],
            lw=1.5, label=f"{name}, back in {rec[name]/FS:.2f} s")
    ax.axvline(rec[name]/FS, color=COLOUR[name], ls=":", lw=1.2)
ax.axvline(0, color="k", ls="--", lw=0.9)
level = np.mean([rest[n]["floor"] for n in FILTERS]) + 3
ax.axhline(level, color="0.55", ls=":", lw=0.9)
ax.text(t_change[keep][-1], level + 0.6, "recovery level, floor + 3 dB", ha="right", va="bottom",
        fontsize=8, color="0.3")
ax.set(xlabel="time from the change [s]", ylabel="misalignment [dB]", ylim=(-25, 8),
       title=r"sKF across $h_o \to -h_o$, AR(-0.9), $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta|$")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="upper right")
plt.show()

**Figure 2.** (a) $v_t$, mean over the realisations, across the change. (b) Share of the realisations
with $\chi_t' < 0$, averaged over 100-step blocks, for the two matched filters (before clipping, for the
clipped one).

In [ ]:
fig, (ax_v, ax_s) = plt.subplots(2, 1, figsize=(9, 6.4), sharex=True, constrained_layout=True)
for name in FILTERS:
    ax_v.plot(t_change[keep], change[name]["v"][keep], color=COLOUR[name], ls=STYLE[name], lw=1.5, label=name)
ax_v.set(ylabel="$v_t$", yscale="log", title="(a) variance per weight, $v_t$")
BLOCK = 100
for name in ("matched", "matched, clipped"):
    share = np.mean(change[name]["slope"] < 0, axis=0)
    idx = np.flatnonzero(keep)
    idx = idx[:len(idx)//BLOCK*BLOCK]
    blocks = share[idx].reshape(-1, BLOCK).mean(axis=1)
    ax_s.plot(t_change[idx].reshape(-1, BLOCK).mean(axis=1), blocks, color=COLOUR[name], ls=STYLE[name],
              lw=1.5, label=name)
ax_s.set(xlabel="time from the change [s]", ylabel="share with $\\chi_t' < 0$", ylim=(0, 1),
         title="(b) how often $\\chi_t' < 0$")
for ax in (ax_v, ax_s):
    ax.axvline(0, color="k", ls="--", lw=0.9)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="upper right")
plt.show()

## 7. Findings

**Checks.** The vectorised quadrature equals notebook 08's `chi_quadrature` to roundoff, and the batched
sKF equals notebook 09's scalar filters. The three filters of notebook 09 land on its $\varepsilon$, steps
at rest (4357, 4756, 3630) and recoveries (7340, 7584, 5979) exactly.

**At rest, clipping changes nothing that matters.** The clipped filter needs a slightly larger
$\varepsilon$ (2.23e-6 against 2.19e-6) and converges in the same 3630 steps. At rest $\chi_t' < 0$ on
about 6 % of the steps.

**After the change, clipping slows the recovery a little, but does not remove the advantage.**

| | minorized | joint | matched | matched, clipped |
|---|---|---|---|---|
| steps to recover | 7340 | 7584 | 5979 | 6312 |
| / joint | 0.97 | 1 | 0.79 | 0.83 |

Clipping costs 333 steps (6 %). That is about a fifth of the matched filter's lead over the joint filter
(1605 steps). So the variance inflation from negative $\chi_t'$ is **a small part** of why the matched sKF
recovers faster, not the main reason. The same holds at every level: clipped / matched is 1.12 at 0 dB and
1.05 to 1.09 below it, while the clipped filter stays ahead of both Laplacian filters from $-10$ dB on.

**Where the rest comes from.** Right after the change, half of the matched filter's $\chi_t'$ are negative
(50 % over the first 500 steps, against 6 % at rest), and the mean $\chi_t'$ falls to about 0.2 (0.33 for
the joint filter, 0.51 for the minorized one). With $\chi_t'$ near zero, eq. (36) barely shrinks
$v_t$ and $\varepsilon$ adds up step after step. So $v_t$ grows even without negative $\chi_t'$: 1.94x its
value at rest for the clipped filter, 2.36x unclipped, against 1.51x (joint) and 1.44x (minorized). The
matched filter reads the large errors of a change as outliers. That makes it trust each one less (slow
start: at 0 dB it is behind both Laplacian filters), but it also stops $v_t$ from shrinking, and the larger $v_t$ then gives
the faster recovery below $-5$ dB. Negative $\chi_t'$ adds to that, but it is not what drives it.

In [ ]:
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")